# TabNet: A Tabular Neural Network for Ground Motion Prediction
- Dataset: dataset/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx
- ANN: Artificial Neural Networks

### Tasks:
- Finding out best hyperparameters for ANN
- Training ANN with the best hyperparameters
- Evaluating the performance of ANN
- Parametric Analysis
- Bin residual plots
- SHAP analysis and feature importance

### Inputs:
- Earth Magnitude
- Joyner-Boore Dist. (km)
- Mechanism Based on Rake Angle 
- Vs30 (m/s) selected for analysis
- Log10 of Vs30 (m/s)
- Log10 of Joyner-Boore Dist. (km)

### Outputs:
- Log10(PGV (cm/sec))
- Log10(PGA (g))
- Log10 of Spectral Acceleration from 0.01s to 4s: (T0.010S - T4.00S)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import time
import copy

print(f"✓ PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")

In [ ]:
import importlib.util
import subprocess
import sys

# Dictionary mapping the 'import' name to the 'pip install' name
dependencies = {
    'shap': 'shap',
    'calamine': 'python-calamine'
    # Add any other weird dependencies from your requirements.txt here!
}

for module, package in dependencies.items():
    # Check if the module is already installed
    if importlib.util.find_spec(module) is None:
        print(f"📦 {package} not found. Installing...")
        # Runs: pip install -q package_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    else:
        print(f"✅ {package} is already installed.")

In [ ]:
# 1. Exact paths for Kaggle cloud vs your local WSL environment
kaggle_path = '/kaggle/input/nga-west2-seismic-data/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx'
local_path = 'dataset/Updated_NGA_West2_Flatfile_RotD50_d050_public_version.xlsx'

# 2. Automatically detect the environment
if os.path.exists('/kaggle/input'):
    print("☁️ Running on Kaggle! Using cloud dataset path.")
    file_path = kaggle_path
else:
    print("💻 Running locally! Using local dataset path.")
    file_path = local_path

# 3. Load the data using the calamine engine
try:
    test_df = pd.read_excel(file_path, nrows=1, engine='calamine')
    print("✅ Columns loaded successfully:\n", test_df.columns)
except FileNotFoundError:
    print(f"❌ ERROR: Could not find the file at {file_path}")

In [ ]:
len(test_df.columns)

In [ ]:
# Helper function to convert period column name to float
def period_to_float(col_name):
    """Convert column name like T0.010S to 0.010"""
    return float(col_name[1:-1])

# Get all column names first
print("Reading column names...")
df_temp = pd.read_excel(file_path, engine='calamine', nrows=0)
all_cols = df_temp.columns.tolist()

# Find period columns <= 4.0 seconds only
period_cols = [col for col in all_cols if col.startswith('T') and col.endswith('S')]
selected_period_cols = [col for col in period_cols if period_to_float(col) <= 4.0]
selected_period_cols_sorted = sorted(selected_period_cols, key=period_to_float)

# Define columns to load
selected_columns = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Vs30 (m/s) selected for analysis',
    'Mechanism Based on Rake Angle',
    'PGA (g)',
    'PGV (cm/sec)'
] + selected_period_cols_sorted

# Define dtypes for faster loading
dtype_dict = {
    'Earthquake Magnitude': 'float32',
    'Joyner-Boore Dist. (km)': 'float32',
    'Vs30 (m/s) selected for analysis': 'float32',
    'Mechanism Based on Rake Angle': 'int8',
    'PGA (g)': 'float32',
    'PGV (cm/sec)': 'float32'
}
for col in selected_period_cols_sorted:
    dtype_dict[col] = 'float32'

print(f"Loading {len(selected_columns)} columns (6 base features + {len(selected_period_cols_sorted)} periods)...")
print(f"Period range: {period_to_float(selected_period_cols_sorted[0]):.3f}s to {period_to_float(selected_period_cols_sorted[-1]):.3f}s")

# Load data
df = pd.read_excel(file_path, engine='calamine', usecols=selected_columns, dtype=dtype_dict)

print(f"✓ Loaded data shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [ ]:
# Data Cleaning: Remove rows with negative, NaN, or Inf values
print("=" * 70)
print("DATA CLEANING")
print("=" * 70)
print(f"\nOriginal shape: {df.shape}")

cols_to_check = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Vs30 (m/s) selected for analysis',
    'Mechanism Based on Rake Angle',
    'PGA (g)',
    'PGV (cm/sec)'
] + selected_period_cols_sorted

# Step 1: Remove rows with any negative values
print("\nRemoving rows with negative values...")
before = len(df)
for col in cols_to_check:
    df = df[df[col] >= 0]
print(f"  After removing negatives: {len(df):,} rows (removed {before - len(df):,})")

# Step 2: Drop NaN / Inf rows
before = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=cols_to_check)
print(f"  After dropping NaN/Inf:   {len(df):,} rows (removed {before - len(df):,})")

df = df.reset_index(drop=True)

print(f"\n✓ Final cleaned shape: {df.shape}")
print(f"\nData ranges:")
print(f"  Earthquake Magnitude:          {df['Earthquake Magnitude'].min():.2f} – {df['Earthquake Magnitude'].max():.2f}")
print(f"  Joyner-Boore Dist. (km):       {df['Joyner-Boore Dist. (km)'].min():.3f} – {df['Joyner-Boore Dist. (km)'].max():.2f}")
print(f"  Vs30 (m/s):                    {df['Vs30 (m/s) selected for analysis'].min():.1f} – {df['Vs30 (m/s) selected for analysis'].max():.1f}")
print(f"  Mechanism Based on Rake Angle: {sorted(df['Mechanism Based on Rake Angle'].unique())}")
print(f"  PGA (g):                       {df['PGA (g)'].min():.6f} – {df['PGA (g)'].max():.4f}")
print(f"  PGV (cm/sec):                  {df['PGV (cm/sec)'].min():.6f} – {df['PGV (cm/sec)'].max():.4f}")
print("=" * 70)


In [ ]:
# Log-transform features and targets, then define input/output columns
print("=" * 70)
print("PREPROCESSING")
print("=" * 70)

df = df.reset_index(drop=True)

# Log10-transformed inputs
df['log10_Rjb']  = np.log10(df['Joyner-Boore Dist. (km)'])
df['log10_Vs30'] = np.log10(df['Vs30 (m/s) selected for analysis'])

# Log10-transformed targets
df['log10_PGA'] = np.log10(df['PGA (g)'])
df['log10_PGV'] = np.log10(df['PGV (cm/sec)'])

print("Creating log10 of spectral periods...")
for col in tqdm(selected_period_cols_sorted, desc="Log-transforming Sa"):
    df[f'log10_{col}'] = np.log10(df[col])

# Column lists used downstream
input_feature_cols = [
    'Earthquake Magnitude',
    'Joyner-Boore Dist. (km)',
    'Mechanism Based on Rake Angle',
    'Vs30 (m/s) selected for analysis',
    'log10_Vs30',
    'log10_Rjb'
]
output_target_cols = (
    ['log10_PGA', 'log10_PGV'] +
    [f'log10_{col}' for col in selected_period_cols_sorted]
)

# Sanity check: no inf/nan in transformed columns
all_transformed = ['log10_Rjb', 'log10_Vs30', 'log10_PGA', 'log10_PGV'] + \
                  [f'log10_{col}' for col in selected_period_cols_sorted]
invalid = df[all_transformed].replace([np.inf, -np.inf], np.nan).isna().any(axis=1).sum()
if invalid:
    print(f"  ⚠ Dropping {invalid} rows with inf/nan after log transform")
    df = df[~df[all_transformed].replace([np.inf, -np.inf], np.nan).isna().any(axis=1)]
    df = df.reset_index(drop=True)
else:
    print("  ✓ All log-transformed values are finite")

print(f"\nFinal dataset: {len(df):,} samples")
print(f"\nInputs  ({len(input_feature_cols)}): {input_feature_cols}")
print(f"Outputs ({len(output_target_cols)}): log10_PGA, log10_PGV + {len(selected_period_cols_sorted)} Sa periods")
print("=" * 70)
